In [12]:
import json
from pathlib import Path
import pandas as pd

root = Path("results") if Path("results").exists() else Path("../results")

rows = []
for path in root.rglob("results.json"):
    parts = path.relative_to(root).parts   # e.g. ('ngram', 'Death', 'lyrics', 'results.json')
    if len(parts) < 4:                     # old runs saved before the text-version folders
        print("skipping", path)
        continue
    r = json.load(open(path))
    rows.append({"model": parts[0], "set": parts[1], "text": parts[2],
                 "accuracy": r["test_accuracy"], "macro_f1": r["test_macro_f1"]})

res = pd.DataFrame(rows)
print(len(res), "runs loaded")
res.groupby(["model", "set"])["text"].apply(sorted)

24 runs loaded


model  set  
bert   Death    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
       Mixed    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
       Power    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
ngram  Death    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
       Mixed    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
       Power    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
Name: text, dtype: object

In [13]:
order = ["lyrics_clean", "lyrics_norm", "lyrics_masked", "lyrics_norm_masked"]
table = res.pivot_table(index=["model", "set"], columns="text", values="macro_f1")[order]
display(table.round(3))
display(table.sub(table["lyrics_clean"], axis=0).round(3))   # change from baseline

text         lyrics_clean  lyrics_norm  lyrics_masked  lyrics_norm_masked
model set                                                                
bert  Death         0.635        0.671          0.623               0.610
      Mixed         0.604        0.525          0.426               0.522
      Power         0.608        0.552          0.553               0.524
ngram Death         0.802        0.820          0.792               0.790
      Mixed         0.737        0.692          0.722               0.650
      Power         0.780        0.744          0.750               0.724

text         lyrics_clean  lyrics_norm  lyrics_masked  lyrics_norm_masked
model set                                                                
bert  Death           0.0        0.036         -0.012              -0.025
      Mixed           0.0       -0.079         -0.178              -0.082
      Power           0.0       -0.056         -0.055              -0.083
ngram Death           0.0        0.018         -0.010              -0.012
      Mixed           0.0       -0.045         -0.015              -0.086
      Power           0.0       -0.036         -0.030              -0.055